# 1. Imports et logs

In [6]:
import os
import logging
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler("pipeline.log"), logging.StreamHandler()],
)
log = logging.getLogger("pipeline")

# 2. Connexion

In [7]:
load_dotenv(override=True)

def get_engine():
    try:
        url = (
            f"postgresql+psycopg2://{os.environ['PG_USER']}:{os.environ['PG_PASSWORD']}"
            f"@{os.environ['PG_HOST']}:{os.environ['PG_PORT']}/{os.environ['PG_DB']}"
        )
        engine = create_engine(url)
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        log.info("Connexion PostgreSQL réussie")
        return engine
    except KeyError as e:
        log.error(f"Variable d'environnement manquante : {e}")
        raise
    except Exception as e:
        log.error(f"Échec de connexion : {e}")
        raise

engine = get_engine()

2026-10-09 14:26:12,526 [INFO] Connexion PostgreSQL réussie


# 3. Extraction

In [8]:
def extract(table):
    df = pd.read_sql(f"SELECT * FROM {table}", engine)
    log.info(f"Table {table} : {len(df)} lignes lues")
    return df

clients_raw = extract("clients")
produits_raw = extract("produits")
commandes_raw = extract("commandes")

2026-10-09 14:26:12,546 [INFO] Table clients : 10 lignes lues
2026-10-09 14:26:12,586 [INFO] Table produits : 10 lignes lues
2026-10-09 14:26:12,593 [INFO] Table commandes : 10 lignes lues


# 4. Aperçu de chaque table

In [9]:
for nom, df in [("clients", clients_raw), ("produits", produits_raw), ("commandes", commandes_raw)]:
    print(f"===== {nom} : {df.shape[0]} lignes, {df.shape[1]} colonnes =====")
    print(df.dtypes)
    display(df.head())

===== clients : 10 lignes, 7 colonnes =====
id_client               str
nom                     str
prenom                  str
email                   str
date_naissance          str
ville                   str
date_creation_compte    str
dtype: object


,id_client,nom,prenom,email,date_naissance,ville,date_creation_compte
0,CL001,Moreau,Julie,julie.moreau@mail.fr,1985-04-12,Lyon,2019-03-01
1,CL002,MOREAU,Julie,j.moreau@mail.fr,NaN,Lyon,2019-03-02
2,CL003,Bernard,Marc,marc.bernard@mail.fr,12/07/1990,Nantes,05-11-2020
3,CL004,Petit,Sophie,sophie.petit@mail.fr,1978-11-30,Lille,2018-06-15
4,CL005,Petit,Sophie,NaN,1978-11-30,Lille,2021-01-20


===== produits : 10 lignes, 5 colonnes =====
id_produit              str
nom_produit             str
categorie               str
prix_ht             float64
stock_disponible    float64
dtype: object


,id_produit,nom_produit,categorie,prix_ht,stock_disponible
0,P001,Canapé Oslo 3 places,Salon,899.0,12.0
1,P002,Canapé Oslo 3 places,salon,899.0,12.0
2,P003,Table basse Malmö,Salon,249.9,34.0
3,P004,Lampadaire Nova,Luminaire,79.9,58.0
4,P005,Lampadaire Nova,Luminaires,79.9,58.0


===== commandes : 10 lignes, 6 colonnes =====
id_commande          str
id_client            str
id_produit           str
date_commande        str
canal                str
montant_ttc      float64
dtype: object


,id_commande,id_client,id_produit,date_commande,canal,montant_ttc
0,CMD001,CL001,P001,2023-05-10,Magasin,1078.80
1,CMD002,CL002,P003,11/05/2023,E-commerce,299.88
2,CMD003,CL004,P004,2023-06-02,Magasin,95.88
3,CMD004,CL006,P006,2023/06/15,E-commerce,78.00
4,CMD005,CL003,P007,NaN,Magasin,648.00


# 5. Taux de valeurs manquantes

In [10]:
tables = {"clients": clients_raw, "produits": produits_raw, "commandes": commandes_raw}

for nom, df in tables.items():
    print(f"--- {nom} ---")
    print(pd.concat([df.isna().sum().rename("nb_manquant"),
                     (df.isna().mean() * 100).round(1).rename("pct_manquant")], axis=1))

--- clients ---
                      nb_manquant  pct_manquant
id_client                       0           0.0
nom                             0           0.0
prenom                          1          10.0
email                           1          10.0
date_naissance                  1          10.0
ville                           0           0.0
date_creation_compte            0           0.0
--- produits ---
                  nb_manquant  pct_manquant
id_produit                  0           0.0
nom_produit                 0           0.0
categorie                   0           0.0
prix_ht                     0           0.0
stock_disponible            1          10.0
--- commandes ---
               nb_manquant  pct_manquant
id_commande              0           0.0
id_client                0           0.0
id_produit               0           0.0
date_commande            1          10.0
canal                    0           0.0
montant_ttc              0           0.0


# 6. Diagnostic de la table clients

In [11]:
# Doublons exacts (en ignorant la clé id_client)
cols_hors_id = [c for c in clients_raw.columns if c != "id_client"]
nb_dup_exact_clients = int(clients_raw.duplicated(subset=cols_hors_id).sum())

# Doublons probables : même nom (casse ignorée) + même prénom
cle = clients_raw["nom"].str.upper() + "|" + clients_raw["prenom"].str.upper()
masque_dup = cle.notna() & cle.duplicated(keep=False)
nb_dup_probables_clients = int((cle.notna() & cle.duplicated()).sum())
display(clients_raw[masque_dup].sort_values("nom", key=lambda s: s.str.upper()))

# Emails invalides (valeurs renseignées uniquement)
regex_email = r"^[\w.+-]+@[\w-]+\.[\w.-]+$"
emails_invalides = clients_raw[clients_raw["email"].notna() & ~clients_raw["email"].str.match(regex_email)]
display(emails_invalides)

# Dates qui ne sont pas au format AAAA-MM-JJ
iso = r"^\d{4}-\d{2}-\d{2}$"
nb_dates_non_iso_clients = 0
for col in ["date_naissance", "date_creation_compte"]:
    non_iso = clients_raw[clients_raw[col].notna() & ~clients_raw[col].str.match(iso)]
    nb_dates_non_iso_clients += len(non_iso)
    print(col, "→ formats non ISO :", len(non_iso))

# Casse incohérente sur le nom
casse_nom = clients_raw[clients_raw["nom"] != clients_raw["nom"].str.title()]
display(casse_nom[["id_client", "nom"]])

,id_client,nom,prenom,email,date_naissance,ville,date_creation_compte
7,CL008,Garcia,Elena,elena.garcia@mail.fr,1988-01-01,Toulouse,2017-05-01
9,CL010,Garcia,Elena,elena.garcia@mail.fr,1988-01-01,Toulouse,2017-05-01
0,CL001,Moreau,Julie,julie.moreau@mail.fr,1985-04-12,Lyon,2019-03-01
1,CL002,MOREAU,Julie,j.moreau@mail.fr,NaN,Lyon,2019-03-02
3,CL004,Petit,Sophie,sophie.petit@mail.fr,1978-11-30,Lille,2018-06-15
4,CL005,Petit,Sophie,NaN,1978-11-30,Lille,2021-01-20


,id_client,nom,prenom,email,date_naissance,ville,date_creation_compte
8,CL009,Roux,Thomas,thomas.roux@mail,1999-12-25,Bordeaux,2023-03-11


date_naissance → formats non ISO : 1
date_creation_compte → formats non ISO : 1


,id_client,nom
1,CL002,MOREAU


# 7. Diagnostic de la table produits

In [12]:
print("Doublons sur id_produit :", produits_raw["id_produit"].duplicated().sum())

# Doublons de contenu (même nom, prix et stock)
sub = ["nom_produit", "prix_ht", "stock_disponible"]
nb_dup_produits = int(produits_raw.duplicated(subset=sub).sum())
display(produits_raw[produits_raw.duplicated(subset=sub, keep=False)])

# Catégories brutes
nb_categories_brutes = produits_raw["categorie"].nunique()
print(produits_raw["categorie"].value_counts())

# Prix aberrants (méthode IQR)
q1, q3 = produits_raw["prix_ht"].quantile([0.25, 0.75])
iqr = q3 - q1
borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
prix_aberrants = produits_raw[(produits_raw["prix_ht"] < borne_basse) | (produits_raw["prix_ht"] > borne_haute)]
print(f"Bornes IQR : [{borne_basse:.2f} ; {borne_haute:.2f}]")
display(prix_aberrants)

# Stocks négatifs
stock_negatif = produits_raw[produits_raw["stock_disponible"] < 0]
display(stock_negatif)

Doublons sur id_produit : 0


,id_produit,nom_produit,categorie,prix_ht,stock_disponible
0,P001,Canapé Oslo 3 places,Salon,899.0,12.0
1,P002,Canapé Oslo 3 places,salon,899.0,12.0
3,P004,Lampadaire Nova,Luminaire,79.9,58.0
4,P005,Lampadaire Nova,Luminaires,79.9,58.0


categorie
Salon             2
salon             1
Luminaire         1
Luminaires        1
Salle à manger    1
salle a manger    1
Déco              1
deco              1
Rangement         1
Name: count, dtype: int64
Bornes IQR : [-1014.13 ; 1903.28]


,id_produit,nom_produit,categorie,prix_ht,stock_disponible
9,P010,Étagère modulable,Rangement,3450.0,15.0


,id_produit,nom_produit,categorie,prix_ht,stock_disponible
5,P006,Chaise Bistrot,Salle à manger,65.0,-3.0


# 8. Diagnostic de la table commandes et intégrité

In [13]:
# Formats de date
formats = {
    "AAAA-MM-JJ": r"^\d{4}-\d{2}-\d{2}$",
    "JJ/MM/AAAA": r"^\d{2}/\d{2}/\d{4}$",
    "AAAA/MM/JJ": r"^\d{4}/\d{2}/\d{2}$",
    "JJ-MM-AAAA": r"^\d{2}-\d{2}-\d{4}$",
}
for nom_f, regex in formats.items():
    print(nom_f, ":", commandes_raw["date_commande"].str.match(regex).sum())
nb_dates_manquantes_cmd = int(commandes_raw["date_commande"].isna().sum())
nb_dates_non_iso_cmd = int(commandes_raw["date_commande"].notna().sum()
                           - commandes_raw["date_commande"].str.match(formats["AAAA-MM-JJ"]).sum())

# Intégrité référentielle
orphelines_client = commandes_raw[~commandes_raw["id_client"].isin(clients_raw["id_client"])]
orphelines_produit = commandes_raw[~commandes_raw["id_produit"].isin(produits_raw["id_produit"])]
print("Commandes sans client :", len(orphelines_client))
print("Commandes sans produit :", len(orphelines_produit))

# Cohérence montant TTC = prix HT x 1,20 (TVA 20 %)
m = commandes_raw.merge(produits_raw[["id_produit", "prix_ht"]], on="id_produit", how="left")
m["montant_attendu"] = (m["prix_ht"] * 1.2).round(2)
incoherents = m[(m["montant_ttc"] - m["montant_attendu"]).abs() > 0.01]
print("Montants incohérents :", len(incoherents))

AAAA-MM-JJ : 6
JJ/MM/AAAA : 1
AAAA/MM/JJ : 1
JJ-MM-AAAA : 1
Commandes sans client : 0
Commandes sans produit : 0
Montants incohérents : 0


# 9. Tableau de synthèse du diagnostic

In [14]:
diagnostic = pd.DataFrame([
    ("clients",   "Valeurs manquantes (toutes colonnes)", int(clients_raw.isna().sum().sum())),
    ("clients",   "Doublons exacts (hors id)",            nb_dup_exact_clients),
    ("clients",   "Doublons probables (nom + prénom)",    nb_dup_probables_clients),
    ("clients",   "Emails invalides",                     len(emails_invalides)),
    ("clients",   "Dates non ISO",                        nb_dates_non_iso_clients),
    ("clients",   "Casse incohérente sur le nom",         len(casse_nom)),
    ("produits",  "Valeurs manquantes (toutes colonnes)", int(produits_raw.isna().sum().sum())),
    ("produits",  "Doublons de contenu",                  nb_dup_produits),
    ("produits",  "Catégories distinctes (brutes)",       nb_categories_brutes),
    ("produits",  "Prix aberrants (IQR)",                 len(prix_aberrants)),
    ("produits",  "Stocks négatifs",                      len(stock_negatif)),
    ("commandes", "Dates manquantes",                     nb_dates_manquantes_cmd),
    ("commandes", "Dates non ISO",                        nb_dates_non_iso_cmd),
    ("commandes", "Commandes orphelines",                 len(orphelines_client) + len(orphelines_produit)),
    ("commandes", "Montants incohérents",                 len(incoherents)),
], columns=["table", "anomalie", "nb"])

display(diagnostic)
log.info("Diagnostic terminé")

,table,anomalie,nb
0,clients,Valeurs manquantes (toutes colonnes),3
1,clients,Doublons exacts (hors id),1
2,clients,Doublons probables (nom + prénom),3
3,clients,Emails invalides,1
4,clients,Dates non ISO,2
5,clients,Casse incohérente sur le nom,1
6,produits,Valeurs manquantes (toutes colonnes),1
7,produits,Doublons de contenu,2
8,produits,Catégories distinctes (brutes),9
9,produits,Prix aberrants (IQR),1


2026-10-09 14:29:21,590 [INFO] Diagnostic terminé


## 3. Transformation

### 3.1 Table clients – règles de nettoyage
| Règle | Choix retenu |
| --- | --- |
| Noms, prénoms, villes | Suppression des espaces, casse normalisée (Première Lettre En Majuscule) |
| Emails | Minuscules ; email au format invalide mis à NULL (jamais corrigé à la main) |
| Dates | Conversion en vrai type date, 4 formats reconnus (AAAA-MM-JJ, JJ/MM/AAAA, AAAA/MM/JJ, JJ-MM-AAAA), lecture jour/mois/année |
| Doublons | Même nom + prénom + ville = même client ; on garde le plus petit id_client et on complète ses NULL avec les valeurs des doublons |
| Valeurs manquantes restantes | Laissées à NULL, sans valeur devinée |
| Commandes liées aux doublons | Dictionnaire id supprimé → id conservé, pour rediriger les commandes à l'étape suivante |

### 3.1.1 Fonction de transformation clients

In [15]:
REGEX_EMAIL = r"^[\w.+-]+@[\w-]+\.[\w.-]+$"
FORMATS_DATE = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"]

def parse_dates(s):
    """Essaie chaque format connu ; ce qui n'est reconnu par aucun reste NaT."""
    resultat = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in FORMATS_DATE:
        resultat = resultat.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    return resultat

def transform_clients(df):
    out = df.copy()
    log.info("clients : %d lignes en entrée", len(out))

    # 1. Normalisation du texte
    for col in ["nom", "prenom", "ville"]:
        out[col] = out[col].str.strip().str.title()
    out["email"] = out["email"].str.strip().str.lower()

    # 2. Emails invalides -> NULL
    invalides = out["email"].notna() & ~out["email"].str.match(REGEX_EMAIL)
    out.loc[invalides, "email"] = None
    log.info("clients : %d email(s) invalide(s) mis à NULL", invalides.sum())

    # 3. Dates -> type datetime
    for col in ["date_naissance", "date_creation_compte"]:
        avant = out[col].notna().sum()
        out[col] = parse_dates(out[col])
        non_reconnues = avant - out[col].notna().sum()
        log.info("clients : %s converti, %d valeur(s) non reconnue(s)", col, non_reconnues)

    # 4. Doublons : on garde le plus petit id et on complète ses NULL
    out = out.sort_values("id_client")
    cle = out["nom"] + "|" + out["prenom"] + "|" + out["ville"]
    out["id_conserve"] = out.groupby(cle)["id_client"].transform("min")
    out["id_conserve"] = out["id_conserve"].fillna(out["id_client"])   # clé inexploitable : id propre

    mapping = out.set_index("id_client")["id_conserve"].to_dict()

    colonnes = [c for c in out.columns if c not in ("id_client", "id_conserve")]
    propre = out.groupby("id_conserve", as_index=False)[colonnes].first()   # first() ignore les NULL
    propre = propre.rename(columns={"id_conserve": "id_client"})

    log.info("clients : %d doublon(s) supprimé(s), %d lignes en sortie",
             len(out) - len(propre), len(propre))
    return propre, mapping

### 3.1.2 Exécution et contrôles clients

In [16]:
clients_clean, map_clients = transform_clients(clients_raw)

display(clients_clean)
print(clients_clean.dtypes)

# Règles de qualité vérifiées automatiquement
assert clients_clean["id_client"].is_unique
assert clients_clean["email"].dropna().str.match(REGEX_EMAIL).all()
assert clients_clean["nom"].equals(clients_clean["nom"].str.title())

# Comparaison avant / après
pd.DataFrame({
    "avant": [len(clients_raw), int(clients_raw.isna().sum().sum())],
    "après": [len(clients_clean), int(clients_clean.isna().sum().sum())],
}, index=["lignes", "valeurs manquantes"])

2026-10-09 14:40:24,269 [INFO] clients : 10 lignes en entrée
2026-10-09 14:40:24,272 [INFO] clients : 1 email(s) invalide(s) mis à NULL
2026-10-09 14:40:24,279 [INFO] clients : date_naissance converti, 0 valeur(s) non reconnue(s)
2026-10-09 14:40:24,282 [INFO] clients : date_creation_compte converti, 0 valeur(s) non reconnue(s)
2026-10-09 14:40:24,289 [INFO] clients : 3 doublon(s) supprimé(s), 7 lignes en sortie


,id_client,nom,prenom,email,date_naissance,ville,date_creation_compte
0,CL001,Moreau,Julie,julie.moreau@mail.fr,1985-04-12,Lyon,2019-03-01
1,CL003,Bernard,Marc,marc.bernard@mail.fr,1990-07-12,Nantes,2020-11-05
2,CL004,Petit,Sophie,sophie.petit@mail.fr,1978-11-30,Lille,2018-06-15
3,CL006,Dubois,Ahmed,ahmed.dubois@mail.fr,2001-02-18,Marseille,2022-07-09
4,CL007,Lefevre,NaN,claire.lefevre@mail.fr,1995-09-05,Paris,2020-02-14
5,CL008,Garcia,Elena,elena.garcia@mail.fr,1988-01-01,Toulouse,2017-05-01
6,CL009,Roux,Thomas,NaN,1999-12-25,Bordeaux,2023-03-11


id_client                          str
nom                                str
prenom                             str
email                              str
date_naissance          datetime64[ns]
ville                              str
date_creation_compte    datetime64[ns]
dtype: object


,avant,après
lignes,10,7
valeurs manquantes,3,2


### 3.2 Table produits – règles de nettoyage
| Règle | Choix retenu |
| --- | --- |
| Noms de produits | Suppression des espaces en trop |
| Catégories | Casse, accents et pluriels ignorés, puis correspondance avec une liste officielle de 5 catégories ; une catégorie inconnue est signalée dans les logs |
| Prix invalides (nul, négatif, manquant) | Ligne écartée |
| Prix atypiques (méthode IQR) | Conservés mais signalés dans une colonne `prix_a_verifier`, car un produit haut de gamme peut être légitime |
| Stocks négatifs | Mis à NULL (on ne sait pas la vraie valeur, et 0 affirmerait une rupture de stock) |
| Doublons | Même nom + catégorie + prix = même produit ; on garde le plus petit id_produit |
| Valeurs manquantes restantes | Laissées à NULL, sans valeur devinée |
| Commandes liées aux doublons | Dictionnaire id supprimé → id conservé, utilisé à l'étape commandes |

### 3.2.1 Fonction de transformation produits

In [17]:
import unicodedata
import numpy as np

CATEGORIES = {
    "salon": "Salon",
    "luminaire": "Luminaire",
    "luminaires": "Luminaire",
    "salle a manger": "Salle à manger",
    "deco": "Déco",
    "rangement": "Rangement",
}

def sans_accents(texte):
    return "".join(c for c in unicodedata.normalize("NFD", texte) if unicodedata.category(c) != "Mn")

def normaliser_categorie(valeur):
    if pd.isna(valeur):
        return np.nan
    cle = sans_accents(valeur).strip().lower()
    return CATEGORIES.get(cle, np.nan)   # NaN si la catégorie n'est pas reconnue

def transform_produits(df):
    out = df.copy()
    log.info("produits : %d lignes en entrée", len(out))

    # 1. Noms de produits
    out["nom_produit"] = out["nom_produit"].str.strip().str.replace(r"\s+", " ", regex=True)

    # 2. Catégories
    brut = out["categorie"]
    out["categorie"] = brut.map(normaliser_categorie)
    inconnues = brut.notna() & out["categorie"].isna()
    if inconnues.any():
        log.warning("produits : catégories non reconnues : %s", brut[inconnues].unique().tolist())

    # 3. Prix invalides (nul, négatif ou manquant) : lignes écartées
    prix_invalide = ~(out["prix_ht"] > 0)
    log.info("produits : %d ligne(s) écartée(s) pour prix invalide", prix_invalide.sum())
    out = out[~prix_invalide]

    # 4. Stocks négatifs -> NULL
    stock = pd.to_numeric(out["stock_disponible"], errors="coerce")
    negatifs = stock < 0
    log.info("produits : %d stock(s) négatif(s) mis à NULL", negatifs.sum())
    out["stock_disponible"] = stock.mask(negatifs).astype("Int64")

    # 5. Doublons : on garde le plus petit id
    out = out.sort_values("id_produit")
    cle = (out["nom_produit"].str.lower().astype("string") + "|"
           + out["categorie"].astype("string") + "|"
           + out["prix_ht"].astype("string"))
    out["id_conserve"] = out.groupby(cle)["id_produit"].transform("min")
    out["id_conserve"] = out["id_conserve"].fillna(out["id_produit"])

    mapping = out.set_index("id_produit")["id_conserve"].to_dict()

    colonnes = [c for c in out.columns if c not in ("id_produit", "id_conserve")]
    propre = out.groupby("id_conserve", as_index=False)[colonnes].first()
    propre = propre.rename(columns={"id_conserve": "id_produit"})

    # 6. Prix atypiques : conservés mais signalés (IQR)
    q1, q3 = propre["prix_ht"].quantile([0.25, 0.75])
    iqr = q3 - q1
    propre["prix_a_verifier"] = (propre["prix_ht"] > q3 + 1.5 * iqr) | (propre["prix_ht"] < q1 - 1.5 * iqr)
    log.info("produits : %d prix atypique(s) signalé(s)", propre["prix_a_verifier"].sum())

    log.info("produits : %d doublon(s) supprimé(s), %d lignes en sortie",
             len(out) - len(propre), len(propre))
    return propre, mapping

### 3.2.2 Exécution et contrôles produits

In [18]:
produits_clean, map_produits = transform_produits(produits_raw)

display(produits_clean)
print(produits_clean.dtypes)

# Règles de qualité vérifiées automatiquement
assert produits_clean["id_produit"].is_unique
assert set(produits_clean["categorie"].dropna()) <= set(CATEGORIES.values())
assert (produits_clean["prix_ht"] > 0).all()
assert (produits_clean["stock_disponible"].dropna() >= 0).all()

# Comparaison avant / après
pd.DataFrame({
    "avant": [len(produits_raw), produits_raw["categorie"].nunique(), int(produits_raw.isna().sum().sum())],
    "après": [len(produits_clean), produits_clean["categorie"].nunique(), int(produits_clean[["nom_produit", "categorie", "prix_ht", "stock_disponible"]].isna().sum().sum())],
}, index=["lignes", "catégories distinctes", "valeurs manquantes"])

2026-10-09 14:47:32,645 [INFO] produits : 10 lignes en entrée
2026-10-09 14:47:32,649 [INFO] produits : 0 ligne(s) écartée(s) pour prix invalide
2026-10-09 14:47:32,650 [INFO] produits : 1 stock(s) négatif(s) mis à NULL
2026-10-09 14:47:32,666 [INFO] produits : 1 prix atypique(s) signalé(s)
2026-10-09 14:47:32,667 [INFO] produits : 2 doublon(s) supprimé(s), 8 lignes en sortie


,id_produit,nom_produit,categorie,prix_ht,stock_disponible,prix_a_verifier
0,P001,Canapé Oslo 3 places,Salon,899.0,12,False
1,P003,Table basse Malmö,Salon,249.9,34,False
2,P004,Lampadaire Nova,Luminaire,79.9,58,False
3,P006,Chaise Bistrot,Salle à manger,65.0,<NA>,False
4,P007,Buffet scandinave,Salle à manger,540.0,7,False
5,P008,Coussin Lin,Déco,19.9,<NA>,False
6,P009,Tapis Berbère,Déco,129.0,22,False
7,P010,Étagère modulable,Rangement,3450.0,15,True


id_produit              str
nom_produit             str
categorie               str
prix_ht             float64
stock_disponible      Int64
prix_a_verifier        bool
dtype: object


,avant,après
lignes,10,8
catégories distinctes,9,5
valeurs manquantes,1,2


### 3.3 Table commandes – règles de nettoyage
| Règle | Choix retenu |
| --- | --- |
| Identifiants liés à des doublons | id_client et id_produit redirigés vers l'identifiant conservé (dictionnaires issus du nettoyage de clients et produits) |
| Dates | Conversion en vrai type date (4 formats reconnus, lecture jour/mois/année) ; date manquante laissée à NULL, la commande est conservée |
| Commandes orphelines | Écartées (client ou produit absent des tables nettoyées) |
| Montants invalides (nul, négatif, manquant) | Ligne écartée |
| Cohérence montant / prix | montant_ttc doit valoir prix_ht x 1,20 (quantité de 1, TVA 20 % : pas de colonne quantité) ; si l'écart dépasse 1 centime, ligne écartée plutôt que corrigée |
| Dates incohérentes | Commande antérieure à la création du compte client, ou dans le futur : écartée |
| Doublons | Même client, produit, date, canal et montant après redirection : une seule commande conservée |

### 3.3.1 Fonction de transformation commandes

In [19]:
TVA = 1.20

def transform_commandes(df, clients_clean, produits_clean, map_clients, map_produits):
    out = df.copy()
    log.info("commandes : %d lignes en entrée", len(out))

    # 1. Texte
    out["canal"] = out["canal"].str.strip()

    # 2. Redirection vers les identifiants conservés
    nouveau_client = out["id_client"].map(map_clients).fillna(out["id_client"])
    nouveau_produit = out["id_produit"].map(map_produits).fillna(out["id_produit"])
    log.info("commandes : %d id_client et %d id_produit redirigés vers l'id conservé",
             (nouveau_client != out["id_client"]).sum(), (nouveau_produit != out["id_produit"]).sum())
    out["id_client"], out["id_produit"] = nouveau_client, nouveau_produit

    # 3. Dates -> type datetime
    avant = out["date_commande"].notna().sum()
    out["date_commande"] = parse_dates(out["date_commande"])
    log.info("commandes : date_commande convertie, %d valeur(s) non reconnue(s), %d manquante(s)",
             avant - out["date_commande"].notna().sum(), out["date_commande"].isna().sum())

    # 4. Commandes orphelines
    orphelines = (~out["id_client"].isin(clients_clean["id_client"])
                  | ~out["id_produit"].isin(produits_clean["id_produit"]))
    log.info("commandes : %d commande(s) orpheline(s) écartée(s)", orphelines.sum())
    out = out[~orphelines]

    # 5. Montants invalides
    invalide = ~(out["montant_ttc"] > 0)
    log.info("commandes : %d montant(s) invalide(s) écarté(s)", invalide.sum())
    out = out[~invalide]

    # 6. Cohérence montant / prix du produit
    out = out.merge(produits_clean[["id_produit", "prix_ht"]], on="id_produit", how="left")
    incoherent = (out["montant_ttc"] - (out["prix_ht"] * TVA).round(2)).abs() > 0.01
    log.info("commandes : %d montant(s) incohérent(s) avec le prix écarté(s)", incoherent.sum())
    out = out[~incoherent]

    # 7. Dates incohérentes (avant la création du compte, ou dans le futur)
    out = out.merge(clients_clean[["id_client", "date_creation_compte"]], on="id_client", how="left")
    incoh_date = ((out["date_commande"] < out["date_creation_compte"])
                  | (out["date_commande"] > pd.Timestamp.today().normalize()))
    log.info("commandes : %d date(s) incohérente(s) écartée(s)", incoh_date.sum())
    out = out[~incoh_date]

    # 8. Doublons (après redirection des id)
    out = out.sort_values("id_commande")
    cols_dup = ["id_client", "id_produit", "date_commande", "canal", "montant_ttc"]
    doublons = out.duplicated(subset=cols_dup)
    log.info("commandes : %d doublon(s) supprimé(s)", doublons.sum())
    out = out[~doublons]

    propre = out[["id_commande", "id_client", "id_produit", "date_commande",
                  "canal", "montant_ttc"]].reset_index(drop=True)
    log.info("commandes : %d lignes en sortie", len(propre))
    return propre

### 3.3.2 Exécution et contrôles commandes

In [20]:
commandes_clean = transform_commandes(commandes_raw, clients_clean, produits_clean,
                                      map_clients, map_produits)

display(commandes_clean)
print(commandes_clean.dtypes)

# Règles de qualité vérifiées automatiquement
assert commandes_clean["id_commande"].is_unique
assert commandes_clean["id_client"].isin(clients_clean["id_client"]).all()
assert commandes_clean["id_produit"].isin(produits_clean["id_produit"]).all()
assert (commandes_clean["montant_ttc"] > 0).all()

# Comparaison avant / après
pd.DataFrame({
    "avant": [len(commandes_raw), int(commandes_raw["date_commande"].isna().sum()),
              round(float(commandes_raw["montant_ttc"].sum()), 2)],
    "après": [len(commandes_clean), int(commandes_clean["date_commande"].isna().sum()),
              round(float(commandes_clean["montant_ttc"].sum()), 2)],
}, index=["lignes", "dates manquantes", "CA TTC total"])

2026-10-09 14:50:08,265 [INFO] commandes : 10 lignes en entrée
2026-10-09 14:50:08,273 [INFO] commandes : 3 id_client et 1 id_produit redirigés vers l'id conservé
2026-10-09 14:50:08,282 [INFO] commandes : date_commande convertie, 0 valeur(s) non reconnue(s), 1 manquante(s)
2026-10-09 14:50:08,285 [INFO] commandes : 0 commande(s) orpheline(s) écartée(s)
2026-10-09 14:50:08,288 [INFO] commandes : 0 montant(s) invalide(s) écarté(s)
2026-10-09 14:50:08,297 [INFO] commandes : 0 montant(s) incohérent(s) avec le prix écarté(s)
2026-10-09 14:50:08,307 [INFO] commandes : 0 date(s) incohérente(s) écartée(s)
2026-10-09 14:50:08,311 [INFO] commandes : 1 doublon(s) supprimé(s)
2026-10-09 14:50:08,315 [INFO] commandes : 9 lignes en sortie


,id_commande,id_client,id_produit,date_commande,canal,montant_ttc
0,CMD001,CL001,P001,2023-05-10,Magasin,1078.80
1,CMD002,CL001,P003,2023-05-11,E-commerce,299.88
2,CMD003,CL004,P004,2023-06-02,Magasin,95.88
3,CMD004,CL006,P006,2023-06-15,E-commerce,78.00
4,CMD005,CL003,P007,NaT,Magasin,648.00
5,CMD006,CL008,P009,2023-07-01,E-commerce,154.80
6,CMD008,CL007,P001,2023-07-20,Magasin,1078.80
7,CMD009,CL009,P010,2023-08-03,E-commerce,4140.00
8,CMD010,CL004,P001,2023-08-22,Magasin,1078.80


id_commande                 str
id_client                   str
id_produit                  str
date_commande    datetime64[ns]
canal                       str
montant_ttc             float64
dtype: object


,avant,après
lignes,10.00,9.00
dates manquantes,1.00,1.00
CA TTC total,8807.76,8652.96


## 4. Jointure et enrichissement des données

Nous créons la table consolidée en joignant les commandes nettoyées avec les tables clients et produits. Nous calculons ensuite les indicateurs clés demandés (CA par produit/catégorie, nombre de commandes par client, etc.).

In [21]:
# 1. Jointure des tables nettoyées
df_consolide = (
    commandes_clean.merge(clients_clean, on="id_client", how="left")
    .merge(produits_clean, on="id_produit", how="left")
)

# 2. Ajout des indicateurs
# Calcul du délai depuis la dernière commande (par rapport à la date la plus récente du jeu de données)
date_max = df_consolide["date_commande"].max()
df_consolide["delai_derniere_commande_jours"] = (date_max - df_consolide["date_commande"]).dt.days

# Nombre de commandes par client
commandes_par_client = df_consolide.groupby("id_client")["id_commande"].transform("nunique")
df_consolide["nb_commandes_client"] = commandes_par_client

# Chiffre d'affaires par catégorie de produit
ca_par_categorie = df_consolide.groupby("categorie")["montant_ttc"].transform("sum")
df_consolide["ca_categorie_total"] = ca_par_categorie

# Chiffre d'affaires par produit
ca_par_produit = df_consolide.groupby("id_produit")["montant_ttc"].transform("sum")
df_consolide["ca_produit_total"] = ca_par_produit

display(df_consolide.head())
print(f"Taille de la table consolidée : {df_consolide.shape}")

,id_commande,id_client,id_produit,date_commande,canal,montant_ttc,nom,prenom,email,date_naissance,...,date_creation_compte,nom_produit,categorie,prix_ht,stock_disponible,prix_a_verifier,delai_derniere_commande_jours,nb_commandes_client,ca_categorie_total,ca_produit_total
0,CMD001,CL001,P001,2023-05-10,Magasin,1078.80,Moreau,Julie,julie.moreau@mail.fr,1985-04-12,...,2019-03-01,Canapé Oslo 3 places,Salon,899.0,12,False,104.0,2,3536.28,3236.40
1,CMD002,CL001,P003,2023-05-11,E-commerce,299.88,Moreau,Julie,julie.moreau@mail.fr,1985-04-12,...,2019-03-01,Table basse Malmö,Salon,249.9,34,False,103.0,2,3536.28,299.88
2,CMD003,CL004,P004,2023-06-02,Magasin,95.88,Petit,Sophie,sophie.petit@mail.fr,1978-11-30,...,2018-06-15,Lampadaire Nova,Luminaire,79.9,58,False,81.0,2,95.88,95.88
3,CMD004,CL006,P006,2023-06-15,E-commerce,78.00,Dubois,Ahmed,ahmed.dubois@mail.fr,2001-02-18,...,2022-07-09,Chaise Bistrot,Salle à manger,65.0,<NA>,False,68.0,1,726.00,78.00
4,CMD005,CL003,P007,NaT,Magasin,648.00,Bernard,Marc,marc.bernard@mail.fr,1990-07-12,...,2020-11-05,Buffet scandinave,Salle à manger,540.0,7,False,NaN,1,726.00,648.00


Taille de la table consolidée : (9, 21)


## 5. Chargement des données (Load)

Nous sauvegardons les données nettoyées (`clients_clean`, `produits_clean`, `commandes_clean`) ainsi que la table consolidée (`df_consolide`) :

1. Dans la base de données PostgreSQL (dans des tables distinctes).
2. En fichiers CSV pour consultation externe.

In [22]:
import os

# 1. Export vers des fichiers CSV
os.makedirs("exports_csv", exist_ok=True)

clients_clean.to_csv("exports_csv/clients_clean.csv", index=False)
produits_clean.to_csv("exports_csv/produits_clean.csv", index=False)
commandes_clean.to_csv("exports_csv/commandes_clean.csv", index=False)
df_consolide.to_csv("exports_csv/table_consolidee.csv", index=False)

print("Exports CSV réalisés avec succès dans le dossier 'exports_csv/'.")

# 2. Sauvegarde dans PostgreSQL (tables avec suffixe '_clean')
try:
    clients_clean.to_sql("client_clean", con=engine, if_exists="replace", index=False)
    produits_clean.to_sql("produit_clean", con=engine, if_exists="replace", index=False)
    commandes_clean.to_sql("commande_clean", con=engine, if_exists="replace", index=False)
    df_consolide.to_sql("table_consolidee", con=engine, if_exists="replace", index=False)
    print("Tables nettoyées enregistrées avec succès dans PostgreSQL.")
except NameError:
    print("Attention : Vérifiez que la variable de connexion SQLAlchemy 'engine' est bien définie.")

Exports CSV réalisés avec succès dans le dossier 'exports_csv/'.
Tables nettoyées enregistrées avec succès dans PostgreSQL.


## 6. Conclusion et bilan du pipeline

- **Clients** : Noms, prénoms et villes normalisés; emails invalides mis à NULL; dates converties. Trois doublons fusionnés, de 10 à 7 clients. Aucune colonne téléphone n'étant présente, les téléphones ne sont pas traités.
- **Produits** : Catégories ramenées aux cinq catégories officielles; deux doublons fusionnés, de 10 à 8 produits. Le stock négatif est mis à NULL. Le prix de P010 (3 450 €) est conservé et signalé par `prix_a_verifier`, sans correction automatique.
- **Commandes** : Trois références client et une référence produit redirigées. CMD007 est supprimée comme doublon après redirection. La date manquante est conservée; aucune commande orpheline, aucun montant invalide ou incohérent, et aucune date incohérente n'a été détectée. Le nombre de commandes passe de 10 à 9 et le CA TTC de 8 807,76 € à 8 652,96 €.
- **Intégrité** : Les assertions d'unicité des identifiants, de validité des références et de positivité des montants passent.
- **Chargement** : Les quatre tables PostgreSQL et les quatre fichiers CSV ont été créés avec succès.